# Libraries

In [1]:
# 1. Base Framework
!pip install -q -U \
    torch==2.11.0 \
    torchvision==0.26.0 \
    torchaudio==2.11.0 \
    --index-url https://download.pytorch.org/whl/cu126

# 2. torchao - force reinstall to actually replace Kaggle's broken copy
!pip install -q --no-deps --force-reinstall torchao \
    --index-url https://download.pytorch.org/whl/cu128

# 3. Evaluation Framework
!pip install -q "lm_eval[hf]"

# 4. Compression Tools
!pip install -q gptqmodel optimum

# 5. Utilities
!pip install -q langdetect --break-system-packages  # for IFEval Task

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 82.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 80.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 226.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 111.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 242.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 236.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 172.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 142.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 117.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 208.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 200.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━

In [2]:
# Libraries
import gc
import glob
import shutil
import os
import random
import subprocess

import numpy as np
import torch
import lm_eval

from dataclasses import dataclass
from typing import Optional
from huggingface_hub import login
from IPython.display import FileLink
from kaggle_secrets import UserSecretsClient

# Global Variables

In [3]:
# --- Identity ---
user_name = "MahmoudOsama20"
email     = "mo2314151@gmail.com"
repo      = "MahmoudOsama20/EdgeCompress"  # Fixed

# --- Model ---
QWEN       = True
model      = "Qwen3-4B"             if QWEN else "Llama-3.2-3B-Instruct"
MODEL_NAME       = "Qwen3-4B-GPTQ"
MODEL_PRETRAINED = "EdgeCompress01/Qwen3-4B-GPTQ-4bit"
MODEL_PRETRAINED_FIX = MODEL_PRETRAINED.replace("/", "__")

# --- Reproducibility ---
SEED = 42

# --- Environment ---
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Classes

In [4]:
@dataclass
class Task:
    key:               str
    name:              str
    category:          str
    num_fewshot:       int  = 0
    batch_size:        int  = 2
    limit:             Optional[int] = None
    apply_chat_template: bool = True
    max_gen_toks:      Optional[int] = None    

# Functions

In [5]:
# REPRODUCIBILITY
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Seeding

In [6]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface login

In [7]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# GitHub login

In [8]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

Logging into GitHub...


# Evaluations

**Configurations**

In [9]:
# Task definition
TASKS = [
    # Math
     # Task("gsm8k",        "gsm8k_cot",                 "math",                num_fewshot=8, batch_size=12,  max_gen_toks=1024),

    # Reasoning
     # Task("arc_challenge","arc_challenge",              "reasoning",           batch_size=16),

    # Instruction Following
    #  Task("ifeval",       "ifeval",                    "instruction_following",batch_size=14,  max_gen_toks=1024),

    # # Perplexity
    #  Task("wikitext",     "wikitext",                  "perplexity",          batch_size=1,  apply_chat_template=False),

    # Knowledge
     Task("mmlu",         "mmlu",                      "knowledge",           num_fewshot=5, batch_size=2,  limit=1400),
]


# Model args
MODEL_ARGS = ",".join([
    f"pretrained={MODEL_PRETRAINED}",
    "dtype=float16",
    "parallelize=True",
    "max_length=4096",
    "trust_remote_code=True",
    "enable_thinking=False",
])

results_dir = f"evaluation_results/{MODEL_NAME}"

**Evaluation loop**

In [10]:
for t in TASKS:
    print(f"\n[{t.category.upper()}] {t.key}")

    cmd = [
        "lm_eval",
        "--model",       "hf",
        "--model_args",  MODEL_ARGS,
        "--tasks",       t.name,
        "--num_fewshot", str(t.num_fewshot),
        "--batch_size",  str(t.batch_size),
        "--seed",        str(SEED),
        "--verbosity", "DEBUG",
        "--output_path", f"{results_dir}/{t.key}",
    ]

    if t.limit is not None:
        cmd += ["--limit", str(t.limit)]
        
    if t.apply_chat_template:
            cmd.append("--apply_chat_template")
        
    if t.max_gen_toks is not None:                             
        cmd += ["--gen_kwargs", f"max_gen_toks={t.max_gen_toks}"]

    subprocess.run(cmd, check=True)
    
    # Push To GitHub
    target_dir_in_repo = f"New_Evaluations/{model}/{MODEL_NAME}/results/{t.key}"
    source_folder = f"{results_dir}/{t.key}/{MODEL_PRETRAINED_FIX}"  # The Kaggle folder you want to upload
    remote_url = f"https://{token}@github.com/{repo}.git"
    
    # A temporary workspace in Kaggle to handle the cloning
    temp_repo_dir = "/kaggle/working/temp_git_repo" 
    # ---------------------

    # 2. Clean up any previous runs and clone the existing repository
    os.system(f"rm -rf {temp_repo_dir}")
    os.system(f"git clone {remote_url} {temp_repo_dir}")
    
    # 3. Create the target directory inside the cloned repo
    full_target_path = f"{temp_repo_dir}/{target_dir_in_repo}"
    os.makedirs(full_target_path, exist_ok=True)
    
    # 4. Copy the contents of your Kaggle folder into the target GitHub folder
    # (Using cp -r to recursively copy all files and subfolders)
    os.system(f"cp -r {source_folder}/* {full_target_path}/")
    
    # 5. Configure, commit, and push the changes
    os.system(f"""
        git -C {temp_repo_dir} config user.email "{email}" && \
        git -C {temp_repo_dir} config user.name "{user_name}" && \
        git -C {temp_repo_dir} add . && \
        git -C {temp_repo_dir} commit -m "Add Kaggle results to {target_dir_in_repo}" && \
        git -C {temp_repo_dir} push origin main
    """)
    
    print(f"Successfully pushed to {repo} inside the '{target_dir_in_repo}' folder!")
    
    torch.cuda.empty_cache()
    gc.collect()
    print("Done")


[KNOWLEDGE] mmlu


2026-04-11:17:53:07 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-04-11:17:53:07 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-11:17:53:13 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
2026-04-11:17:53:15 INFO     [evaluator:211] Setting random seed to 42 | Setting numpy seed to 42 | Setting torch manual seed to 42 | Setting fewshot manual seed to 42
2026-04-11:17:53:15 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'EdgeCompress01/Qwen3-4B-GPTQ-4bit', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096, 'trust_remote_code': True, 'enable_thinking': False}
2026-04-11:17:53:22 INFO     [models.huggingface:178] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
2026-04-11:17:53:23 DEBUG    [models.huggingface:537] Using model type 'causal'
2026-04-11:17:53:26 INFO    


{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pr

fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
INFO:httpx:HTTP Request: HEAD https://huggingface.co/EdgeCompress01/Qwen3-4B-GPTQ-4bit/resolve/main/model.safetensors "HTTP/1.1 302 Found"


{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 6.0.3
Transformers : 5.5.3
Torch        : 2.11.0+cu126
Triton       : 3.6.0
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}


...

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/cais/mmlu/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/cais/mmlu/c30699e8356da336a370243923dbaf21066bb9fe/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/cais/mmlu/c30699e8356da336a370243923dbaf21066bb9fe/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/cais/mmlu/resolve/c30699e8356da336a370243923dbaf21066bb9fe/mmlu.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/cais/mmlu/cais/mmlu.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/cais/mmlu/revision/c30699e8356da336a370243923dbaf21066bb9fe "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/cais/mmlu/resolve/c30699e8356da336a370243923dbaf21

hf ({'pretrained': 'EdgeCompress01/Qwen3-4B-GPTQ-4bit', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096, 'enable_thinking': False}), gen_kwargs: ({}), limit: 1400.0, num_fewshot: 5, batch_size: 2
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.6316|±  |0.0039|
| - humanities                          |      2|none  |      |acc   |↑  |0.5454|±  |0.0069|
|  - formal_logic                       |      1|none  |     5|acc   |↑  |0.6190|±  |0.0434|
|  - high_school_european_history       |      1|none  |     5|acc   |↑  |0.7091|±  |0.0355|
|  - high_school_us_history             |      1|none  |     5|acc   |↑  |0.7647|±  |0.0298|
|  - high_school_world_history          |      1|none  |     5|acc   |↑  |0.8101|±  |0.0255|
|  - international_law                  |      1

Cloning into '/kaggle/working/temp_git_repo'...


[main 041ce7a] Add Kaggle results to New_Evaluations/Qwen3-4B/Qwen3-4B-GPTQ/results/mmlu
 1 file changed, 4376 insertions(+)
 create mode 100644 New_Evaluations/Qwen3-4B/Qwen3-4B-GPTQ/results/mmlu/results_2026-04-11T20-55-13.744092.json
Successfully pushed to MahmoudOsama20/EdgeCompress inside the 'New_Evaluations/Qwen3-4B/Qwen3-4B-GPTQ/results/mmlu' folder!
Done


To https://github.com/MahmoudOsama20/EdgeCompress.git
   04cae91..041ce7a  main -> main


# Save reports

In [11]:
zip_path = shutil.make_archive(
    "evaluation_results",   # zip file name
    "zip",                  # format
    "evaluation_results"    # folder to compress
)